In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

In [16]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")

# ID mappings (from 02_id_encoder.ipynb)
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
user_id_2idx_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_id_2idx_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_2idx_path = os.path.join(id_mappings_dir, "scene_2idx.json")
id_vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

# Precomputed embeddings
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")               # (N_items, 768)
image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base.pt")             # (N_items, 768)
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")              # (N_interactions, D_time)
time_feature_cols_path = os.path.join(preprocessed_dir, "time_features_columns.json")      # list of column names

# fusion artifacts from this notebook will be saved here
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
os.makedirs(fusion_artifacts_dir, exist_ok=True)

In [8]:
# sanity check
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("id_mappings_dir:", id_mappings_dir)
print("text_embeddings_path:", text_embeddings_path)
print("image_embeddings_path:", image_embeddings_path)
print("time_features_path:", time_features_path)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
id_mappings_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings
text_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base.pt
time_features_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [11]:
# Next, we will load Embedding Sizes (Dynamic Sizing)
# Load shapes without pulling everything into GPU memory
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
text_dim = text_embeddings.shape[1]

image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_dim = image_embeddings.shape[1]

time_features = torch.load(time_features_path, map_location="cpu")
time_raw_dim = time_features.shape[1]

In [19]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("\n--- Embedding/Feature Dimensions ---")
print("Text embedding dim:", text_dim)
print("Image embedding dim:", image_dim)
print("Time raw feature dim:", time_raw_dim)
print("Time feature columns:", time_feature_cols)


--- Embedding/Feature Dimensions ---
Text embedding dim: 1536
Image embedding dim: 768
Time raw feature dim: 5
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [20]:
# Load vocab sizes (for ID embeddings)
with open(id_vocab_config_path, "r") as f:
    id_vocab_config = json.load(f)

user_vocab_size = int(id_vocab_config["num_users"])
item_vocab_size = int(id_vocab_config["num_items"])
scene_vocab_size = int(id_vocab_config["num_scenes"])

print("\n--- ID Vocabulary Sizes ---")
print("user_vocab_size:", user_vocab_size)
print("item_vocab_size:", item_vocab_size)
print("scene_vocab_size:", scene_vocab_size)


--- ID Vocabulary Sizes ---
user_vocab_size: 67615
item_vocab_size: 59142
scene_vocab_size: 5


In [22]:
!git status

Refresh index: 100% (14/14), done.
On branch main
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    notebooks/01_image_path_validation.ipynb
	modified:   notebooks/06_fusion_module.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/03a_image_path_validation.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
